###Knowledge Base Construction, Alignment, and Expansion

1.Norbert Dias DEVARAJ

2.Venkat Sai KADARI

###STEP 1.1 — Fetch Data from OpenAlex

First, the requests library is imported to handle HTTP requests. An empty list called all_authors is created to store the retrieved author data.

In [ ]:
import requests

all_authors = []

for page in range(1, 41):  # 4 pages → ~1000 authors
    url = f"https://api.openalex.org/authors?search=physics&per-page=25&page={page}"
    data = requests.get(url).json()
    all_authors.extend(data["results"])

print("Total authors:", len(all_authors))
print(all_authors[0])

Total authors: 1000
{'id': 'https://openalex.org/A5100405516', 'orcid': 'https://orcid.org/0000-0002-2957-7319', 'display_name': 'Y. Bai', 'display_name_alternatives': ['Bai, Y', 'Bai, Y.', 'Bai, Y. X.', 'Bai, Yang', 'Bai, Yang [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]; Berger, Joshua [Univ. of Pittsburgh, PA (United States). Dept. of Physics and Astronomy]', 'Bai, Yang [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]; Berger, Joshua [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]; Osborne, James [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]; Stefanek, Ben A. [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]', 'Bai, Yang [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]; Korwar, Mrunal [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]', 'Bai, Yang [Univ. of Wisconsin, Madison, WI (United States). Dept. of Physics]; Korwar, Mrunal [Univ. of Wiscons


This code retrieves author data from the OpenAlex API using keyword-based search ("physics").
It iterates through multiple pages of results, collects author information, and stores it in a list.
Finally, it prints the total number of authors fetched and displays a sample record.

prepares the environment for working with RDF data by installing the required library.

STEP 1.2 — Build RDF Graph

In [ ]:
!pip install rdflib

In [ ]:

from rdflib import Graph, Namespace, Literal, RDF

g = Graph()

EX = Namespace("http://example.org/")
g.bind("ex", EX)

for author in all_authors:
    if "display_name" not in author:
        continue

    name = author["display_name"].replace(" ", "_")

    author_uri = EX[name]

    # Type
    g.add((author_uri, RDF.type, EX.Researcher))

    # Works count
    if "works_count" in author:
        g.add((author_uri, EX.worksCount, Literal(author["works_count"])))

    # Concepts (fields)
    if "x_concepts" in author:
        for concept in author["x_concepts"][:2]:  # limit to avoid noise
            concept_name = concept["display_name"].replace(" ", "_")
            g.add((author_uri, EX.fieldOfWork, EX[concept_name]))

print("Triples so far:", len(g))

Triples so far: 3943



This code constructs an RDF knowledge graph using rdflib from the collected author data.
Each author is represented as a unique URI and assigned the type "Researcher".
Additional properties such as number of works and research fields are added as triples.
The graph is progressively built and the total number of triples is displayed.

###STEP 2.1 — Install + Imports

In [ ]:
!pip install requests tqdm pandas

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
import time

STEP 2.2 — Wikidata Search Function

In [ ]:
def search_wikidata(name):
    url = "https://www.wikidata.org/w/api.php"

    headers = {
        "User-Agent": "Mozilla/5.0 (Colab Project)"
    }

    params = {
        "action": "wbsearchentities",
        "search": name,
        "language": "en",
        "format": "json"
    }

    for _ in range(3):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=5)

            if response.status_code == 200:
                data = response.json()

                if "search" in data and len(data["search"]) > 0:
                    result = data["search"][0]
                    return result["id"], 0.8

        except:
            pass

        time.sleep(1)

    return None, 0.0


This function searches for a given entity name in Wikidata using its API.
It sends a request and retrieves the best matching result, returning the entity ID along with a confidence score.
A retry mechanism is included to handle request failures and improve robustness.

STEP 2.3 — Perform Entity Linking

In [ ]:
mapping = []
not_found = 0

for author in tqdm(all_authors[:800]):
    if "display_name" not in author:
        continue

    name = author["display_name"]

    # clean name
    name_clean = name.split(",")[0]
    local_name = name_clean.replace(" ", "_")

    wikidata_id, confidence = search_wikidata(name_clean)

    if wikidata_id:
        mapping.append({
            "PrivateEntity": f"ex:{local_name}",
            "WikidataURI": f"wd:{wikidata_id}",
            "Confidence": confidence
        })

        print(name_clean, "→", wikidata_id)

    else:
        # handle not found entity
        g.add((EX[local_name], RDF.type, EX.Researcher))
        not_found += 1

    time.sleep(0.2)

print("\n Total aligned entities:", len(mapping))
print(" Not found entities:", not_found)

  0%|          | 1/800 [00:00<06:44,  1.97it/s]

Y. Bai → Q57710315


  0%|          | 2/800 [00:00<06:01,  2.21it/s]

M. Lattanzi → Q53969758


  0%|          | 3/800 [00:01<05:39,  2.35it/s]

...

Physics Dep → Q97958922


100%|██████████| 800/800 [37:54<00:00,  2.39s/it]

Basic Research Community for Physics → Q116971468


100%|██████████| 800/800 [37:54<00:00,  2.84s/it]


 Total aligned entities: 206
 Not found entities: 594



This code aligns local author entities with Wikidata by searching each author name using the Wikidata API.
If a match is found, it stores the mapping along with a confidence score; otherwise, the author is kept as a local entity.
It also tracks the number of successfully aligned and unmatched entities while adding delays to avoid API rate limits.

step 2.4 — Create Mapping Table

In [ ]:
df = pd.DataFrame(mapping)

df.head()

,PrivateEntity,WikidataURI,Confidence
0,ex:Y._Bai,wd:Q57710315,0.8
1,ex:M._Lattanzi,wd:Q53969758,0.8
2,ex:L._Demortier,wd:Q53444110,0.8
3,ex:Shankar_Balasubramanian,wd:Q7488670,0.8
4,ex:John_Terning,wd:Q112473259,0.8


In [ ]:
df.to_csv("mapping.csv", index=False)

step 2.5 — Create Alignment RDF File

In [ ]:
from rdflib import Graph, Namespace, URIRef

alignment_graph = Graph()

EX = Namespace("http://example.org/")

for row in mapping:
    subject = EX[row["PrivateEntity"].replace("ex:", "")]
    object_uri = URIRef("http://www.wikidata.org/entity/" + row["WikidataURI"].replace("wd:", ""))

    alignment_graph.add((
        subject,
        URIRef("http://www.w3.org/2002/07/owl#sameAs"),
        object_uri
    ))

print("Alignment triples:", len(alignment_graph))

Alignment triples: 206



This code creates an alignment graph that links local entities to corresponding Wikidata entities.
For each mapped author, it generates an `owl:sameAs` relationship between the local URI and the Wikidata URI.
This enables semantic alignment between the local knowledge graph and external knowledge sources.

In [ ]:
alignment_graph.serialize("alignment.ttl", format="turtle")

<Graph identifier=N483003d7104d40a0b72f8d4e6b53aaa6 (<class 'rdflib.graph.Graph'>)>

###STEP 3 — Predicate Alignment

defines predicate alignment using Turtle (TTL) format. It maps custom properties from the example namespace to equivalent properties in Wikidata using the owl:equivalentProperty relation.

In [ ]:
ttl_content = """EX = Namespace("http://example.org/") .
@prefix wdt: <http://www.wikidata.org/prop/direct/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:fieldOfWork owl:equivalentProperty wdt:P101 .

ex:worksCount owl:equivalentProperty wdt:P800 .
"""

with open("predicate_alignment.ttl", "w") as f:
    f.write(ttl_content)

###STEP 4 — KB EXPANSION

4.1 Extract Wikidata QIDs
First, the Turtle content defining property mappings is written to a file, enabling alignment between local RDF properties and Wikidata properties.

In [ ]:
wikidata_ids = [row["WikidataURI"].replace("wd:", "") for row in mapping]

print("Aligned entities:", len(wikidata_ids))

Aligned entities: 206


4.2 SPARQL 1-Hop Expansion

In [ ]:
def get_1hop_triples(qid, limit=2000):
    url = "https://query.wikidata.org/sparql"

    query = f"""
    SELECT ?p ?o WHERE {{
      wd:{qid} ?p ?o .
      FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
    }}
    LIMIT {limit}
    """

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "KB-Expansion"
    }

    try:
        response = requests.get(url, params={"query": query}, headers=headers, timeout=15)
        data = response.json()

        triples = []
        for item in data["results"]["bindings"]:
            s = f"http://www.wikidata.org/entity/{qid}"
            p = item["p"]["value"]
            o = item["o"]["value"]

            # skip empty / bad values
            if not p or not o:
                continue

            triples.append((s, p, o))

        return triples

    except Exception as e:
        return []


This function retrieves 1-hop triples for a given Wikidata entity using a SPARQL query.
It fetches direct relationships (properties and objects) associated with the entity and filters only valid Wikidata predicates.
The results are processed into triples and returned for further knowledge graph expansion.

2-HOP EXPANSION

In [ ]:
def get_2hop_triples(qid, limit=4500):
    url = "https://query.wikidata.org/sparql"

    query = f"""
SELECT ?mid ?p ?o WHERE {{
  wd:{qid} ?p1 ?mid .
  ?mid ?p ?o .

  FILTER(isIRI(?mid))   #  VERY IMPORTANT

  FILTER(STRSTARTS(STR(?p1), "http://www.wikidata.org/prop/direct/"))
  FILTER(STRSTARTS(STR(?p), "http://www.wikidata.org/prop/direct/"))
}}
LIMIT {limit}
"""

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "KB-Expansion"
    }

    try:
        response = requests.get(url, params={"query": query}, headers=headers, timeout=15)
        data = response.json()

        triples = []
        for item in data["results"]["bindings"]:
            s = item["mid"]["value"]
            p = item["p"]["value"]
            o = item["o"]["value"]

            # skip bad values
            if not s or not p or not o:
                continue

            triples.append((s, p, o))

        return triples

    except Exception as e:
        return []


This function retrieves 2-hop triples from Wikidata by first finding intermediate entities connected to a given entity, and then extracting their relationships.
It uses a SPARQL query to explore deeper connections in the graph while ensuring only valid URI-based entities and direct properties are included.
The resulting triples help expand the knowledge graph with richer contextual information.

4.3 Build Expanded Graph

In [ ]:
from rdflib import Graph, URIRef, Literal
from tqdm import tqdm

expanded_graph = Graph()

for qid in tqdm(wikidata_ids[:180]):

    triples_1hop = get_1hop_triples(qid)
    triples_2hop = get_2hop_triples(qid)

    for triples in [triples_1hop, triples_2hop]:
        for s, p, o in triples:
            try:
                subject = URIRef(s)
                predicate = URIRef(p)

                if o.startswith("http"):
                    obj = URIRef(o)
                else:
                    obj = Literal(o)

                expanded_graph.add((subject, predicate, obj))
            except:
                continue

    time.sleep(0.3)

100%|██████████| 180/180 [04:46<00:00,  1.59s/it]


4.4 Predicate Filtering
It performs predicate filtering to reduce noise and keep only the most important relationships in the expanded graph.

In [ ]:
from collections import Counter

predicate_counts = Counter()

for s, p, o in expanded_graph:
    predicate_counts[p] += 1

# keep top 120 (NOT 100)
top_predicates = set([p for p, _ in predicate_counts.most_common(120)])

filtered_graph = Graph()

for s, p, o in expanded_graph:
    if p in top_predicates:
        filtered_graph.add((s, p, o))

print("After predicate filtering:", len(filtered_graph))

After predicate filtering: 50542


4.5 Entity Filtering
It performs entity filtering to further refine the graph by removing less important nodes.

First, the frequency (degree) of each entity is calculated by counting how many times it appears as a subject or object in the filtered graph. A minimum degree threshold of 4 is defined.

In [ ]:
from collections import Counter

entity_count = Counter()

for s, p, o in filtered_graph:
    entity_count[s] += 1
    entity_count[o] += 1

min_degree = 4

final_graph = Graph()

for s, p, o in filtered_graph:
    # FIX: only filter SUBJECT (not both)
    if entity_count[s] >= min_degree:
        final_graph.add((s, p, o))

print("After entity filtering:", len(final_graph))

After entity filtering: 50491


4.6 Final Statistics
It iterates through all triples in the final graph and collects unique entities (subjects and objects) and relations (predicates) using sets. This ensures that duplicates are not counted.

In [ ]:
entities = set()
relations = set()

for s, p, o in final_graph:
    entities.add(s)
    entities.add(o)
    relations.add(p)

print("\nFINAL KB STATS")
print("Triples:", len(final_graph))
print("Entities:", len(entities))
print("Relations:", len(relations))


FINAL KB STATS
Triples: 50491
Entities: 36006
Relations: 120


In [ ]:
print(len(wikidata_ids))

206


In [ ]:
expanded_graph.serialize("expanded_kb.nt", format="nt")

/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N73234a6ad537410799fee0f953d87b38 (<class 'rdflib.graph.Graph'>)>

In [ ]:
import json

with open("alignment.json", "w") as f:
    json.dump(mapping, f, indent=2)

##  Preparation for Knowledge Graph Embedding

The constructed knowledge base is designed to support downstream tasks in the next lab session, particularly Knowledge Graph Embedding (KGE) and link prediction.

To ensure compatibility with embedding models such as TransE, DistMult, and ComplEx, several considerations were incorporated during KB construction:

* **Sufficient Graph Size:** The KB contains over 50,000 triples, which provides enough data for learning meaningful embeddings.
* **Controlled Number of Relations:** The number of relations was limited to 120, ensuring that embedding models can effectively learn relation-specific patterns.
* **Entity Coverage:** The KB includes a large number of entities, enabling diverse interactions and improving generalization in embedding models.
* **Graph Connectivity:** The use of both 1-hop and 2-hop expansion ensures that the graph is well-connected, reducing isolated components and improving embedding quality.
* **Noise Reduction:** Predicate filtering and removal of literal-heavy triples improve signal quality for embedding algorithms.

In the next lab session, this KB will be used for:

* Splitting into training, validation, and test sets
* Training embedding models such as TransE, DistMult, and ComplEx
* Evaluating link prediction performance
* Comparing model effectiveness

The quality and structure of the current KB are expected to directly influence embedding performance, making careful construction and cleaning essential for downstream success.
